**_`incremental data load`_**

In [0]:

#dbutils.widgets.text("src_array","")
dbutils.widgets.text("src","")
src_value = dbutils.widgets.get("src")

In [0]:

#src_value=dbutils.widgets.get("src_array")
#src_value
display(src_value)

In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","csv")\
    .option("cloudFiles.schemaLocation",f"/Volumes/workspace/bronze/bronze_volume/{src_value}/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","rescue")\
    .load(f"/Volumes/workspace/raw/raw_volume/raw_source/{src_value}")

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .trigger(once=True)\
    .option("checkpointLocation",f"/Volumes/workspace/bronze/bronze_volume/{src_value}/checkpoint")\
    .option("path",f"/Volumes/workspace/bronze/bronze_volume/{src_value}/data")\
    .start()

In [0]:
%sql
select * from delta.`/Volumes/workspace/bronze/bronze_volume/bookings/data`;

An operation is idempotent if:
Running it once or running it 10 times → the data ends up the same.

In Databricks workflows:
Jobs may retry automatically if they fail.
Pipelines may be re-run manually.
Streaming jobs may restart.
Schedulers may trigger overlapping runs.
If your logic is not idempotent, you may get:
Duplicate records
Double updates
Corrupted tables

In [0]:
%sql
select * from delta.`/Volumes/workspace/bronze/bronze_volume/flights/data/`;